In [1]:
from XGBoostModel import XGBoostModel
from RunData import RunPipeline
import pandas as pd
import optuna
from sklearn.preprocessing import LabelEncoder
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold ,train_test_split, cross_val_score
import kagglehub
import os
import zipfile
from seed_utils import SEED
import requests


/home/daniel7/.conda/envs/my_env_311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# GenericDataPipeline

In [2]:
class GenericDataPipeline:

    def rank_features(self, X, y, n_folds=5):
        print(f"Starting feature importance calculation with XGBoost and {n_folds}-fold CV...")

        X_copy = X.copy()

        null_ratios = {}
        for col in X_copy.columns:
            null_ratios[col] = X_copy[col].isna().mean()

        for col in X_copy.select_dtypes(include=['int64', 'float64']).columns:
            X_copy[col] = X_copy[col].replace([np.inf, -np.inf], np.nan)
            
            non_null = X_copy[col].dropna()
            if len(non_null) > 0:
                mean_val = non_null.mean()
                std_val = non_null.std()
                if std_val > 0 and not np.isnan(mean_val):
                    upper_bound = mean_val + 5 * std_val
                    lower_bound = mean_val - 5 * std_val
                    X_copy[col] = X_copy[col].clip(lower=lower_bound, upper=upper_bound)
        
        num_cols = X_copy.select_dtypes(include=['int64', 'float64']).columns
        cat_cols = X_copy.select_dtypes(include=['object']).columns
        
        X_processed = X_copy.copy()
        
        for col in cat_cols:
            if X_processed[col].isna().sum() > 0:
                most_freq = X_processed[col].mode()[0] if not X_processed[col].isna().all() else 'Unknown'
                X_processed[col] = X_processed[col].fillna(most_freq)
            X_processed[col] = X_processed[col].astype('category')
        
        for col in num_cols:
            if X_processed[col].isna().sum() > 0:
                median_val = X_processed[col].median() if not X_processed[col].isna().all() else 0
                X_processed[col] = X_processed[col].fillna(median_val)
        
        X_filled = X_processed

        mean_target = float(np.mean(y))
        print(f"Mean target value (for base_score): {mean_target:.6f}")

        base_score = 0.5
        if 0 < mean_target < 1:
            base_score = mean_target

        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'auc',
            'max_depth': 6,
            'eta': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_weight': 3,
            'alpha': 0.01,
            'lambda': 1.0,
            'gamma': 0.1,
            'base_score': base_score, 
            'seed': 42,
            'tree_method': 'hist',
            'device': 'cuda'

        }

        kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

        fold_importances = []

        fold_scores = []

        print(f"Training XGBoost models across {n_folds} folds...")

        y_values = set(y.unique())
        print(f"Unique target values: {y_values}")

        if not all(isinstance(val, (int, float)) for val in y_values):
            print("Warning: Target variable contains non-numeric values. Converting to numeric.")
            y = y.astype(float)

        if not all(val in [0, 1] for val in y_values):
            print("Warning: Target variable contains values other than 0 and 1.")
            print("Converting target to binary: 0 for negative class, 1 for positive class.")
            y = (y > 0).astype(int)

        # Perform cross-validation
        for fold, (train_idx, val_idx) in enumerate(kf.split(X_filled, y)):
            print(f"Fold {fold+1}/{n_folds}")

            X_train, X_val = X_filled.iloc[train_idx], X_filled.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            train_pos = np.mean(y_train)
            val_pos = np.mean(y_val)
            print(f"  Train positive rate: {train_pos:.4f}, Validation positive rate: {val_pos:.4f}")

            try:
                dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
                dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

                # Train model
                model = xgb.train(
                    params,
                    dtrain,
                    num_boost_round=100,
                    early_stopping_rounds=20,
                    evals=[(dval, 'validation')],
                    verbose_eval=False
                )

                y_pred = model.predict(dval)
                score = roc_auc_score(y_val, y_pred)
                fold_scores.append(score)

                try:
                    importance = model.get_score(importance_type='gain')

                    fold_importance = {col: importance.get(col, 0) for col in X_filled.columns}

                    max_value = max(fold_importance.values()) if max(fold_importance.values()) > 0 else 1
                    normalized_importance = {col: val/max_value for col, val in fold_importance.items()}

                    fold_importances.append(normalized_importance)
                except Exception as e:
                    print(f"Warning: Could not get feature importance for fold {fold+1}: {str(e)}")

            except Exception as e:
                print(f"Error in fold {fold+1}: {str(e)}")
                print("Skipping this fold and continuing with others.")

        if fold_scores:
            mean_auc = np.mean(fold_scores)
            print(f"Mean AUC across {len(fold_scores)} successful folds: {mean_auc:.4f}")
        else:
            print("No successful folds. Cannot calculate mean AUC.")

        if not fold_importances:
            print("No feature importance information could be gathered from any fold.")
            print("Using a simple fallback method for feature scoring.")

            avg_importance = {}
            for col in X_filled.columns:
                try:
                    valid_mask = ~pd.isna(X_copy[col]) & ~pd.isna(y)
                    if valid_mask.sum() > 10: 
                        if X_filled[col].dtype.name == 'category':
                            col_values = X_filled[col][valid_mask].cat.codes
                        else:
                            col_values = X_filled[col][valid_mask]
                        
                        y_values = y[valid_mask]
                        corr = abs(np.corrcoef(col_values, y_values)[0, 1])
                        avg_importance[col] = corr if not np.isnan(corr) else 0
                    else:
                        avg_importance[col] = np.random.uniform(0, 0.001)
                except Exception as e:
                    print(f"Warning: Could not calculate correlation for {col}: {str(e)}")
                    avg_importance[col] = np.random.uniform(0, 0.001)
        else:
            avg_importance = {}
            for col in X_filled.columns:
                importances = [fold_imp.get(col, 0) for fold_imp in fold_importances]
                if importances:
                    avg_importance[col] = np.mean(importances)
                else:
                    null_ratio = null_ratios.get(col, 0)
                    avg_importance[col] = max(0.001 * (1 - null_ratio), 0.0001)  

        feature_data = []
        for col in X_copy.columns:
            gain_score = avg_importance.get(col, 0)
            null_ratio = null_ratios.get(col, 0)
            feature_data.append({
                'feature_name': col,
                'gain_score': gain_score,
                'null_ratio': null_ratio
            })
        
        feature_df = pd.DataFrame(feature_data)
        feature_df = feature_df.sort_values('gain_score', ascending=False)
        
        print("\nFeature scores:")
        print("-" * 100)
        print(f"{'Rank':<5} | {'Feature':<30} | {'Gain Score':<15} | {'Null Ratio':<10}")
        print("-" * 100)

        for i, row in feature_df.iterrows():
            rank = i + 1
            feat = row['feature_name']
            gain = row['gain_score']
            null_ratio = row['null_ratio']
            print(f"{rank:<5} | {feat:<30} | {gain:.6f} | {null_ratio:.4f}")

        return feature_df

    def objective(self, trial, feature_scores_with_nulls, all_features_score, df, label="readmitted"):
        K = 10
        selected_features = feature_scores_with_nulls['feature_name'].to_list()[:K]
        print("selected_features")
        print(selected_features)
        all_features = all_features_score['feature_name'].to_list()
        # Create binary assignment for each of the selected features:
        assignment = {}
        for feat in selected_features:
            # 1 means feature goes to stage1, 0 means feature is used in stage2
            assignment[feat] = trial.suggest_categorical(f"assign_{feat}", [1, 0])
        
        for feat in all_features:
            if feat not in assignment.keys():
                assignment[feat] = 1
        vals = 0
        for key,value in assignment.items():
            vals +=value
        if vals==0:
            return 99999
        penalty = 0 
        # Derive feature sets based on the assignment:
        stage1_features = [feat for feat, assign in assignment.items() if assign == 1]
        stage2_features = [feat for feat, assign in assignment.items() if assign == 0]
        csv_name = f"trial_{trial.number}_results.csv"
        dm = RunPipeline()
        validation_loss = dm.full_run(df,stage1_features,stage2_features,label,csv_name)
        if validation_loss == 999:
            return validation_loss
        else:
            final_objective = validation_loss + penalty
            return final_objective
    def preprocessing(self,df):
        for col in df.columns:
            if df[col].dtype == 'object':
                # Replace ? with NaN
                df[col] = df[col].replace(['?', ''], np.nan)
                # Convert to categorical codes
                if df[col].isna().sum() < len(df):  # If not all values are NaN
                    df[col] = pd.Categorical(df[col]).codes

                    # Ensure -1 (missing) values are converted to NaN
                    df[col] = df[col].replace(-1, np.nan)
        # Convert boolean columns to int
        for col in df.select_dtypes(include=['bool']).columns:
            df[col] = df[col].astype(int)
        return df 
    def credit_risk_scoring(self,n_trials=20):
        zip_path = "credit-risk-scoring-with-xgboost2023 (4).zip"
        extract_path = "credit_data"
        os.makedirs(extract_path, exist_ok=True)
        
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        data1 = pd.read_csv('credit_data/data_devsample.csv')
        data2 = pd.read_csv('credit_data/data_to_score.csv')
        df = pd.merge(data1, data2, on='SK_ID_CURR', how='inner')
        df.replace([np.inf, -np.inf], np.nan, inplace=True)

        nan_columns = df.columns[df.isna().any()].tolist()
        columns_to_remove = [col for col in df.columns if col.endswith('_y') or col in ['TIME_x', 'BASE_x', 'DAY_x', 'MONTH_x']]
        df = df.drop(columns=columns_to_remove, errors='ignore')
        df = self.preprocessing(df)
        return df

# Credit risk scoring with xgboost

In [12]:
fs = GenericDataPipeline()
study = fs.credit_risk_scoring(20)


Using 'TARGET' as target variable
Starting feature importance calculation with XGBoost and 5-fold CV...
Mean target value (for base_score): 0.080200
Training XGBoost models across 5 folds...
Unique target values: {np.float64(0.0), np.float64(1.0)}
Fold 1/5
  Train positive rate: 0.0802, Validation positive rate: 0.0802
Fold 2/5
  Train positive rate: 0.0802, Validation positive rate: 0.0802
Fold 3/5
  Train positive rate: 0.0802, Validation positive rate: 0.0802
Fold 4/5
  Train positive rate: 0.0802, Validation positive rate: 0.0802
Fold 5/5
  Train positive rate: 0.0802, Validation positive rate: 0.0803


[I 2025-05-17 16:47:29,759] A new study created in memory with name: no-name-90a10587-2cfa-4567-b7ba-d707474fd63d


Mean AUC across 5 successful folds: 0.7423

Feature scores:
----------------------------------------------------------------------------------------------------
Rank  | Feature                        | Gain Score      | Null Ratio
----------------------------------------------------------------------------------------------------
43    | EXT_SOURCE_3_x                 | 0.994056 | 0.1992
42    | EXT_SOURCE_2_x                 | 0.875086 | 0.0021
13    | NAME_EDUCATION_TYPE_x          | 0.466165 | 0.0000
51    | FLOORSMAX_AVG_x                | 0.442927 | 0.4972
41    | EXT_SOURCE_1_x                 | 0.422664 | 0.5644
3     | CODE_GENDER_x                  | 0.378248 | 0.0000
79    | FLOORSMAX_MEDI_x               | 0.375767 | 0.4972
63    | ELEVATORS_MODE_x               | 0.346577 | 0.5323
31    | REGION_RATING_CLIENT_W_CITY_x  | 0.323544 | 0.0000
10    | AMT_GOODS_PRICE_x              | 0.321276 | 0.0008
23    | FLAG_EMP_PHONE_x               | 0.316128 | 0.0000
97    | FLAG_DOCUME

[I 2025-05-17 16:47:30,339] A new study created in memory with name: no-name-cf39e470-16d4-4550-9ba6-4b2906b8476c
/sise/home/daniel7/IncrementalLearing/Changes /XGBoostModel.py:21: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
[I 2025-05-17 16:47:50,352] Trial 0 finished with value: 0.7292833494553115 and parameters: {'max_depth': 10, 'learning_rate': 0.008837247408349572, 'n_estimators': 893, 'min_child_weight': 5, 'subsample': 0.7091463158071074, 'colsample_bytree': 0.8011262701407743, 'gamma': 1.6292878772378816, 'lambda': 5.8950575234896885, 'alpha': 1.3596284083115031, 'scale_pos_weight': 8.809587352858307}. Best is trial 0 with value: 0.7292833494553115.


********** run results **********
{'max_depth': 10, 'learning_rate': 0.008837247408349572, 'n_estimators': 893, 'min_child_weight': 5, 'subsample': 0.7091463158071074, 'colsample_bytree': 0.8011262701407743, 'gamma': 1.6292878772378816, 'lambda': 5.8950575234896885, 'alpha': 1.3596284083115031, 'scale_pos_weight': 8.809587352858307, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.7296113754493717), np.float64(0.7214498392364052), np.float64(0.7367888336801575)]
********** run results **********


...

********** run results **********
{'max_depth': 3, 'learning_rate': 0.017215563938526877, 'n_estimators': 531, 'min_child_weight': 3, 'subsample': 0.8739662463536735, 'colsample_bytree': 0.7012693375376614, 'gamma': 4.996466469986258, 'lambda': 7.400480522624221, 'alpha': 1.1206962616268188, 'scale_pos_weight': 2.7503547219309206, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.7384185439170926), np.float64(0.7324134427010331), np.float64(0.749347190277899)]
********** run results **********


/sise/home/daniel7/IncrementalLearing/Changes /XGBoostModel.py:21: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
[I 2025-05-17 20:33:22,102] Trial 19 finished with value: 0.7283881960091834 and parameters: {'max_depth': 10, 'learning_rate': 0.036752554104119504, 'n_estimators': 184, 'min_child_weight': 5, 'subsample': 0.8165077644915055, 'colsample_bytree': 0.6015937658981625, 'gamma': 3.84009075007068, 'lambda': 1.0982606619900521, 'alpha': 2.6359174958522726, 'scale_pos_weight': 4.137909138072299}. Best is trial 4 with value: 0.7065741185611673.


********** run results **********
{'max_depth': 10, 'learning_rate': 0.036752554104119504, 'n_estimators': 184, 'min_child_weight': 5, 'subsample': 0.8165077644915055, 'colsample_bytree': 0.6015937658981625, 'gamma': 3.84009075007068, 'lambda': 1.0982606619900521, 'alpha': 2.6359174958522726, 'scale_pos_weight': 4.137909138072299, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.7288476448405584), np.float64(0.7216017431279833), np.float64(0.7347152000590085)]
********** run results **********
Best parameters found: {'max_depth': 6, 'learning_rate': 0.007114176283404116, 'n_estimators': 201, 'min_child_weight': 6, 'subsample': 0.8212760012697019, 'colsample_bytree': 0.9985705124377553, 'gamma': 4.627718252523839, 'lambda': 1.6209083607207044, 'alpha': 2.6274228480615194, 'scale_pos_weight': 0.31323098058968846}
Best CV AUC score: 0.7066

===== Detailed Cross-Validation Results with Best Parameters =====
Params: {'max_depth': 6, 'lea

[I 2025-05-17 20:33:59,648] Trial 19 finished with value: -0.013735436269180057 and parameters: {'assign_EXT_SOURCE_3_x': 1, 'assign_FLOORSMAX_AVG_x': 0, 'assign_EXT_SOURCE_1_x': 1, 'assign_FLOORSMAX_MEDI_x': 0, 'assign_ELEVATORS_MODE_x': 1, 'assign_MEDIAN_AMTCR_0M_6M_x': 0, 'assign_STD_AMTCR_0M_6M_x': 0, 'assign_MIN_AMTCR_0M_6M_x': 1, 'assign_ELEVATORS_MEDI_x': 0, 'assign_DAYS_BIRTH_x': 1}. Best is trial 0 with value: -0.3839893566599263.



Base model (no extended)
AUC: 0.7143, Accuracy: 0.9122, F1 Score: 0.0000

Extended model (with extended)
AUC: 0.7303, Accuracy: 0.9174, F1 Score: 0.1387

Combined model (no extended)
AUC: 0.7183, Accuracy: 0.9122, F1 Score: 0.0000

Combined model (with extended)
AUC: 0.7126, Accuracy: 0.9259, F1 Score: 0.0000

Results Summary:
                            Model       AUC  Accuracy       F1  \
0        Base model (no extended)  0.714347  0.912153  0.00000   
1  Extended model (with extended)  0.730301  0.917399  0.13866   
2    Combined model (no extended)  0.718289  0.912153  0.00000   
3  Combined model (with extended)  0.712625  0.925851  0.00000   

                                                                                                                                                                                                                                                                                                                                                   

# Final Results

AUC Summary:
                         Model      AUC
      Base model (no extended) 0.678989
Extended model (with extended) 0.723547
  Combined model (no extended) 0.665925
Combined model (with extended) 0.695310

Base Features:
FLOORSMAX_AVG_x, FLOORSMAX_MEDI_x, MEDIAN_AMTCR_0M_6M_x, ..., MEAN_AMTCR_OVERDUE_0M_INFM_TYPE_EQ_CLOSED_DIV_MEAN_AMTCR_0M_INFM_TYPE_EQ_CLOSED_x

Extended Features:
EXT_SOURCE_3_x, EXT_SOURCE_1_x, ELEVATORS_MODE_x, STD_AMTCR_0M_6M_x, has_extended

Target - TARGET 

AUC Differences:
Extended model - Combined (with extended): 0.028237000000000068
Base model - Combined (no extended):       0.013063999999999965

Standard Deviations:
Extended model - Std Dev: 0.0038
Base model - Std Dev:     0.0044
